In [1]:
from dotenv import load_dotenv

load_dotenv()


True

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

model = ChatOpenAI(
    model="qwen3.5-flash",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    extra_body={
        "thinking": {
            "type": "disabled" # 关闭推理
        }
    },
    # api_key 走 OPENAI_API_KEY 环境变量
)

In [3]:
ai = model.invoke([
    SystemMessage("你是一个翻译助手"),
    HumanMessage("把'Agent 是 LLM 的应用形态'翻成英文"),
])
print(ai.content)

**Agents are the application form of LLMs.**

*(Other natural variations depending on context)*
*   *The Agent is the application manifestation of the LLM.*
*   *Agents represent the primary application paradigm of LLMs.*
*   *The Agent serves as the application form for LLMs.*


In [6]:
prompts = [
    [HumanMessage("20个字解释 Token")],
    [HumanMessage("20个字解释 Embedding")],
    [HumanMessage("20个字解释 Function Calling")],
]

# 一次性把 3 个请求并发出去，框架内部帮你管协程池
results = model.batch(prompts, config={"max_concurrency": 3})
for r in results:
    print("-", r.content)

- Token 是数字世界的凭证，用于验证身份或代表资产。
- 将离散数据映射为连续向量，保留语义相似性。
- 让大模型调用外部工具获取实时数据或执行特定操作。


In [7]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个{role}。"),
    MessagesPlaceholder("history"),       # ← 历史消息插槽
    ("human", "{question}"),
])

history = [
    HumanMessage("我叫小明"),
    AIMessage("你好小明！"),
]

msgs = prompt.invoke({
    "role": "客服",
    "history": history,
    "question": "我刚才说我叫什么？",
}).to_messages()

for m in msgs:
    print(type(m).__name__, "-", m.content)

SystemMessage - 你是一个客服。
HumanMessage - 我叫小明
AIMessage - 你好小明！
HumanMessage - 我刚才说我叫什么？
